# Run the Current Gurobi Model

IEOR 4004 Team 34 — municipal parking plus resident-suggested curbside charging.

Run cells from top to bottom in Google Colab. This notebook runs the shared Python implementation; it does not contain a second optimization model. The default scenario is a hypothetical $15M budget, a 0.5-mile radius and at most 150 four-port curbside sites. A valid Gurobi license is required.


## 1. Install dependencies and load the repository

In [ ]:
%pip install -q pandas numpy openpyxl geopy gurobipy matplotlib

import os, subprocess, sys, json
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path('/content/IEOR4004-project-Energy-Team34')
if PROJECT_ROOT.exists():
    subprocess.run(['git', '-C', str(PROJECT_ROOT), 'pull', '--ff-only', 'origin', 'main'], check=True)
else:
    subprocess.run(['git', 'clone', 'https://github.com/Hanruitang20/IEOR4004-project-Energy-Team34.git', str(PROJECT_ROOT)], check=True)
os.chdir(PROJECT_ROOT)
os.environ['LC_ALL'] = 'C'
print('Git commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())


## 2. Configure the license

If your runtime already has a working license, skip this cell. Otherwise add `GRB_WLSACCESSID`, `GRB_WLSSECRET` and `GRB_LICENSEID` to Colab Secrets and allow notebook access. Values are read into a temporary runtime license file and are not printed or committed. A local Mac academic license may not be usable on a separate Colab machine.


In [ ]:
import os
from pathlib import Path

secret_names = ["GRB_WLSACCESSID", "GRB_WLSSECRET", "GRB_LICENSEID"]
loaded = []
try:
    from google.colab import userdata
    for name in secret_names:
        try:
            value = userdata.get(name)
        except Exception:
            value = None
        if value:
            os.environ[name] = str(value)
            loaded.append(name)
except ImportError:
    pass

if len(loaded) == 3:
    license_path = Path("/tmp/gurobi.lic")
    license_path.write_text(
        f"WLSACCESSID={os.environ['GRB_WLSACCESSID']}\n"
        f"WLSSECRET={os.environ['GRB_WLSSECRET']}\n"
        f"LICENSEID={os.environ['GRB_LICENSEID']}\n"
    )
    os.environ["GRB_LICENSE_FILE"] = str(license_path)
    print("Loaded Gurobi WLS credentials from Colab Secrets into a temporary runtime license file.")
elif loaded:
    print("Loaded only:", ", ".join(loaded))
else:
    print("No Colab WLS secrets loaded. Continue only if another valid Gurobi license is available.")

## 3. Run Gurobi

In [ ]:
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'colab_current_model'
subprocess.run([sys.executable, 'run_gurobi.py', '--output-dir', str(OUTPUT_DIR)], cwd=PROJECT_ROOT, check=True)


## 4. Review the solution

The recorded main reference serves about 67.64% of projected demand for $11.97M, adding 41 municipal ports and 460 curbside ports. Nonzero solver gaps and alternative portfolios can change a fresh result. Check actual solver gaps and feasibility rather than requiring identical site selections.


In [ ]:
from IPython.display import display
summary = json.loads((OUTPUT_DIR / 'summary.json').read_text())
checks = json.loads((OUTPUT_DIR / 'sanity_checks.json').read_text())
fields = ['primary_solver_status', 'solver_status', 'primary_mip_gap', 'final_mip_gap', 'service_rate', 'investment_cost_usd', 'budget_slack_usd', 'selected_municipal_sites', 'additional_ports', 'selected_curbside_sites', 'curbside_ports']
display(pd.DataFrame({'value': {k: summary[k] for k in fields}}))
display(pd.DataFrame({'value': checks}))
for filename in ['municipal_candidate_solution.csv', 'curbside_candidate_solution.csv']:
    plan = pd.read_csv(OUTPUT_DIR / filename)
    print(filename)
    display(plan.loc[plan['selected'].eq(1)])


## 5. Run checks and download outputs

In [ ]:
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests'], cwd=PROJECT_ROOT, check=True)
import shutil
archive = shutil.make_archive('/content/Team34_Gurobi_Results', 'zip', root_dir=OUTPUT_DIR)
from google.colab import files
files.download(archive)


## Optional sensitivity analysis

From the project terminal, run `python run_resident_sensitivity.py --include-radius`, then `python plot_resident_sensitivity.py results/sensitivity_resident`. This replaces the corresponding recorded sensitivity outputs; copy that folder first to preserve them. See [the run guide](https://github.com/Hanruitang20/IEOR4004-project-Energy-Team34/blob/main/GUROBI_RUN_INSTRUCTIONS.md).
